# Stack Overflow tag prediction from titles: ML vs DL proof of concept

<a href="https://colab.research.google.com/github/lrondal/poc-nlp/blob/main/notebooks/stackoverflow_tag_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Goal**: given only the **title** of a Stack Overflow **post**, predict its **tags**.
Two approaches are compared on exactly the same data split and metrics:

| | Approach |
|---|---|
| **ML model** | TF-IDF (word + character n-grams) → One-vs-Rest logistic regression |
| **DL model** | Fine-tuned `roberta-base` with a multi-label (sigmoid) head (**v2**, see section 6) |
| **Ensemble** | Weighted average of ML and DL tag scores |

Vocabulary used throughout follows [`GLOSSARY.md`](../GLOSSARY.md).

> Run on Google Colab with a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lrondal/poc-nlp.git"  # edit if the repo lives elsewhere

if IN_COLAB:
    if not os.path.exists("/content/poc-nlp"):
        !git clone -q {REPO_URL} /content/poc-nlp
    %cd /content/poc-nlp
    # scikit-learn, pandas, torch and transformers are preinstalled on Colab.

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

SEED = 42
MIN_POSTS_PER_TAG = 20  # a tag becomes a "kept tag" if it appears on at least this many posts

random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = next(p for p in [Path("data/stackoverflow_posts.csv"), Path("../data/stackoverflow_posts.csv")] if p.exists())

## 2. Load the data

The CSV has one row per *(post, tag)* pair. We regroup it into one row per **post** with its **title** and list of **tags**.
`tag_position` is only used to keep the original order; it carries no "main tag" meaning.

In [ ]:
raw = pd.read_csv(DATA_PATH, keep_default_na=False)  # keeps the tag "null" as text
print(f"{len(raw):,} rows, {raw.post_id.nunique():,} posts, {raw.tag_name.nunique():,} distinct tags")
raw.head()

In [ ]:
posts = (
    raw.sort_values(["post_id", "tag_position"])
    .groupby("post_id")
    .agg(title=("title", "first"), tags=("tag_name", list))
    .reset_index()
)
posts.head()

## 3. Exploratory analysis

In [ ]:
tag_counts = posts.explode("tags")["tags"].value_counts()
tags_per_post = posts["tags"].str.len()
title_words = posts["title"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
tag_counts.head(30).plot.bar(ax=axes[0], title="30 most frequent tags (posts per tag)")
tags_per_post.value_counts().sort_index().plot.bar(ax=axes[1], title="Tags per post")
title_words.plot.hist(bins=30, ax=axes[2], title="Words per title")
plt.tight_layout()
plt.show()

print(f"Tags with >= {MIN_POSTS_PER_TAG} posts: {(tag_counts >= MIN_POSTS_PER_TAG).sum()} / {len(tag_counts)}")
print(f"Tags with exactly 1 post: {(tag_counts == 1).sum()}")
print(f"Longest title: {title_words.max()} words")

The tag distribution is extremely long-tailed: most tags appear on only a handful of posts and cannot be learned
from ~4.6k examples. We therefore restrict prediction to **kept tags**.

## 4. Kept tags, label sets and data split

* **Kept tag**: a tag present on at least `MIN_POSTS_PER_TAG` posts.
* **Label set**: the kept tags of a post. Posts with an empty label set are dropped.
* Split **by post**, 80 / 10 / 10 (train / validation / test), fixed seed. Both models use the exact same split.

In [ ]:
kept_tags = sorted(tag_counts[tag_counts >= MIN_POSTS_PER_TAG].index)
kept = set(kept_tags)

posts["label_set"] = posts["tags"].apply(lambda tags: [t for t in tags if t in kept])
dataset = posts[posts["label_set"].str.len() > 0].reset_index(drop=True)
print(f"{len(kept_tags)} kept tags; {len(dataset):,} / {len(posts):,} posts keep a non-empty label set")

mlb = MultiLabelBinarizer(classes=kept_tags)
Y = mlb.fit_transform(dataset["label_set"]).astype(np.float32)

idx_train, idx_tmp = train_test_split(np.arange(len(dataset)), test_size=0.2, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED)

X_train, X_val, X_test = (dataset["title"].iloc[i].tolist() for i in (idx_train, idx_val, idx_test))
Y_train, Y_val, Y_test = Y[idx_train], Y[idx_val], Y[idx_test]
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

### Shared evaluation

Each model outputs a **tag score** in [0, 1] for every kept tag. **Predicted tags** = tags whose score ≥ a threshold,
plus the best-scored tag if nothing passes (every post has at least one tag). Thresholds are tuned on the
**validation** set, then frozen for the **test** set. A threshold is either **global** (one value for all tags,
maximising micro-F1) or **per tag** (one value per kept tag, maximising that tag's F1).

| Metric | Meaning |
|---|---|
| **micro-F1** | F1 over all (post, tag) decisions: dominated by frequent tags. Headline metric. |
| **macro-F1** | Average of per-tag F1: every kept tag counts equally, so rare tags matter. |
| **P@1** | Is the single best-scored tag in the label set? |
| **R@3** | Share of the label set found among the 3 best-scored tags. |
| **tags/post** | Average number of predicted tags per post. Compare with the true average (≈1.6). |

In [ ]:
def predicted_tags_matrix(scores: np.ndarray, threshold) -> np.ndarray:
    """threshold: one float for all tags, or an array with one threshold per kept tag."""
    pred = (scores >= threshold).astype(int)
    pred[np.arange(len(scores)), scores.argmax(axis=1)] = 1  # always at least one tag
    return pred


def tune_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    grid = np.arange(0.05, 0.95, 0.01)
    f1s = [f1_score(y_true, predicted_tags_matrix(scores, t), average="micro", zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])


def tune_thresholds_per_tag(y_true: np.ndarray, scores: np.ndarray, global_threshold: float, min_positives: int = 5) -> np.ndarray:
    """One threshold per tag. A tag keeps the global threshold if it has too few validation examples
    or if its own threshold does not beat the global one on its F1 (avoids over-predicting weak tags)."""
    thresholds = np.full(scores.shape[1], global_threshold)
    grid = np.arange(0.05, 0.95, 0.01)
    for j in np.flatnonzero(y_true.sum(axis=0) >= min_positives):
        f1s = [f1_score(y_true[:, j], scores[:, j] >= t, zero_division=0) for t in grid]
        global_f1 = f1_score(y_true[:, j], scores[:, j] >= global_threshold, zero_division=0)
        if max(f1s) > global_f1:
            thresholds[j] = grid[int(np.argmax(f1s))]
    return thresholds


def evaluate(y_true: np.ndarray, scores: np.ndarray, threshold) -> dict:
    pred = predicted_tags_matrix(scores, threshold)
    top1 = scores.argmax(axis=1)
    top3 = np.argsort(-scores, axis=1)[:, :3]
    hits_at_3 = np.take_along_axis(y_true, top3, axis=1).sum(axis=1)
    return {
        "micro_f1": f1_score(y_true, pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "p@1": y_true[np.arange(len(y_true)), top1].mean(),
        "r@3": (hits_at_3 / y_true.sum(axis=1)).mean(),
        "tags/post": pred.sum(axis=1).mean(),
        "threshold": float(threshold) if np.isscalar(threshold) else "per-tag",
    }


def top_tags(scores_row: np.ndarray, threshold, k: int = 5) -> list[tuple[str, float]]:
    thresholds = np.broadcast_to(threshold, scores_row.shape)
    order = np.argsort(-scores_row)[:k]
    chosen = [i for i in order if scores_row[i] >= thresholds[i]] or [order[0]]
    return [(kept_tags[i], round(float(scores_row[i]), 3)) for i in chosen]


print(f"True tags per post on test: {Y_test.sum(axis=1).mean():.2f}")

## 5. ML model: TF-IDF + One-vs-Rest logistic regression

* **Word** n-grams (1–2) with a token pattern that keeps programming names intact (`c#`, `c++`, `.net`, `asp.net`, `ruby-on-rails`).
* **Character** n-grams (2–5) to be robust to spelling variants and unseen words.
* One logistic regression per kept tag. `C` (inverse regularisation) is picked on the validation set.

Unchanged from v1, so it serves as the fixed reference.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer

TOKEN_PATTERN = r"(?u)\.?[\w#+]+(?:[.\-][\w#+]+)*"


def make_ml_model(C: float):
    features = make_union(
        TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    )
    classifier = OneVsRestClassifier(LogisticRegression(C=C, max_iter=2000), n_jobs=-1)
    return make_pipeline(features, classifier)


ml_start = time.perf_counter()
ml_search = {}
for C in [1, 5, 10, 30]:
    candidate = make_ml_model(C).fit(X_train, Y_train)
    val_scores = candidate.predict_proba(X_val)
    ml_search[C] = (evaluate(Y_val, val_scores, tune_threshold(Y_val, val_scores))["micro_f1"], candidate)
    print(f"C={C:<3} val micro-F1={ml_search[C][0]:.3f}")

best_C = max(ml_search, key=lambda c: ml_search[c][0])
ml_model = ml_search[best_C][1]
ml_train_time = time.perf_counter() - ml_start  # includes the C search
print(f"Best C={best_C}; total training time {ml_train_time:.1f}s")

In [ ]:
ml_val_scores = ml_model.predict_proba(X_val)
ml_threshold = tune_threshold(Y_val, ml_val_scores)
ml_test_scores = ml_model.predict_proba(X_test)
ml_results = evaluate(Y_test, ml_test_scores, ml_threshold)
pd.Series(ml_results, name="ML model (test)")

## 6. DL model v2: fine-tuned RoBERTa

The v1 run (DistilBERT) tied the ML model on frequent tags, lost on rare tags, and rarely predicted more than one tag.
v2 changes target each observed symptom:

| v1 symptom | v2 change |
|---|---|
| Lowercasing and an English-only vocabulary blur `C#`, `C++`, `.NET` | **`roberta-base`**: case-sensitive byte-level tokenizer, bigger model (125M vs 66M parameters) |
| First 4 epochs stuck predicting "no tag" (val micro-F1 ≈ 0.1) | **Separate learning rates**: 5e-4 for the new classification head, 3e-5 for the pre-trained encoder |
| Scores of secondary tags squashed to 0.3–0.45 → one tag per post | **`pos_weight` in the BCE loss** (√(negatives/positives) per tag, capped at 10), so missing a true tag costs more than predicting a wrong one |
| Still improving at the last epoch (20) | **Up to 30 epochs**, early stopping with patience 4 |
| One global threshold for tags with very different score ranges | **Per-tag thresholds** (tags with ≥ 5 validation examples), reported next to the global one |
| Bigger model = slower | **Mixed precision (fp16)** on GPU, ~2× faster |

`DL_MODEL_NAME` can be switched to `"microsoft/codebert-base"` (RoBERTa pre-trained on code) or back to
`"distilbert-base-uncased"` to isolate the effect of the training changes from the model change.

In [ ]:
import copy

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DL_MODEL_NAME = "roberta-base"  # alternatives: "microsoft/codebert-base", "distilbert-base-uncased" (v1)
MAX_LEN = 48          # titles are short; see "Longest title" in section 3
BATCH_SIZE = 32
ENCODER_LR = 3e-5     # pre-trained layers: small steps
HEAD_LR = 5e-4        # new, randomly initialised classification head: bigger steps
MAX_EPOCHS = 30
PATIENCE = 4
POS_WEIGHT_CAP = 10.0

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device, "|", torch.cuda.get_device_name(0) if use_amp else "no GPU: this will be slow")

tokenizer = AutoTokenizer.from_pretrained(DL_MODEL_NAME)


def make_loader(titles, labels, shuffle):
    enc = tokenizer(list(titles), padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels, dtype=torch.float32))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader = make_loader(X_train, Y_train, shuffle=True)
val_loader = make_loader(X_val, Y_val, shuffle=False)
test_loader = make_loader(X_test, Y_test, shuffle=False)

print("Tokenisation example:", tokenizer.tokenize("Cannot compile C++ code with .NET and C#"))

In [ ]:
@torch.no_grad()
def dl_scores(model, loader) -> np.ndarray:
    model.eval()
    out = []
    for input_ids, attention_mask, _ in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device)).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)


dl_model = AutoModelForSequenceClassification.from_pretrained(
    DL_MODEL_NAME,
    num_labels=len(kept_tags),
    problem_type="multi_label_classification",
    id2label=dict(enumerate(kept_tags)),
    label2id={t: i for i, t in enumerate(kept_tags)},
    ignore_mismatched_sizes=True,  # replace any existing classification head with an 86-tag one
).to(device)

# Positive-class weights: rare tags get a bigger reward for being found
positives = Y_train.sum(axis=0)
pos_weight = np.clip(np.sqrt((len(Y_train) - positives) / positives), 1.0, POS_WEIGHT_CAP)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=device))
print(f"pos_weight: min {pos_weight.min():.1f}, median {np.median(pos_weight):.1f}, max {pos_weight.max():.1f}")

# Two parameter groups: pre-trained encoder vs new classification head
encoder_ids = {id(p) for p in dl_model.base_model.parameters()}
optimizer = torch.optim.AdamW(
    [
        {"params": [p for p in dl_model.parameters() if id(p) in encoder_ids], "lr": ENCODER_LR},
        {"params": [p for p in dl_model.parameters() if id(p) not in encoder_ids], "lr": HEAD_LR},
    ],
    weight_decay=0.01,
)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

history, best_f1, best_state, epochs_without_gain = [], -1.0, None, 0
dl_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dl_model.train()
    running_loss = 0.0
    for input_ids, attention_mask, labels in train_loader:
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = dl_model(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device)).logits
        loss = loss_fn(logits.float(), labels.to(device))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(dl_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()

    val_scores = dl_scores(dl_model, val_loader)
    val_eval = evaluate(Y_val, val_scores, tune_threshold(Y_val, val_scores))
    history.append({
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_micro_f1": val_eval["micro_f1"],
        "val_tags_per_post": val_eval["tags/post"],
    })
    print(f"epoch {epoch:2d}  train loss {history[-1]['train_loss']:.4f}  "
          f"val micro-F1 {val_eval['micro_f1']:.3f}  val tags/post {val_eval['tags/post']:.2f}")

    if val_eval["micro_f1"] > best_f1:
        best_f1, epochs_without_gain = val_eval["micro_f1"], 0
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in dl_model.state_dict().items()})
    else:
        epochs_without_gain += 1
        if epochs_without_gain >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs")
            break

dl_train_time = time.perf_counter() - dl_start
dl_model.load_state_dict(best_state)
best_epoch = max(history, key=lambda h: h["val_micro_f1"])["epoch"]
print(f"Best val micro-F1 {best_f1:.3f} at epoch {best_epoch}; training time {dl_train_time:.1f}s")

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
fig, ax1 = plt.subplots(figsize=(8, 4))
hist["train_loss"].plot(ax=ax1, color="tab:blue", label="train loss")
ax2 = ax1.twinx()
hist["val_micro_f1"].plot(ax=ax2, color="tab:orange", label="val micro-F1")
ax1.set_ylabel("train loss"); ax2.set_ylabel("val micro-F1")
fig.legend(loc="upper center"); plt.title("DL model v2 training"); plt.show()

In [ ]:
dl_val_scores = dl_scores(dl_model, val_loader)
dl_threshold = tune_threshold(Y_val, dl_val_scores)
dl_tag_thresholds = tune_thresholds_per_tag(Y_val, dl_val_scores, dl_threshold)
dl_test_scores = dl_scores(dl_model, test_loader)

dl_results = evaluate(Y_test, dl_test_scores, dl_threshold)
dl_results_per_tag = evaluate(Y_test, dl_test_scores, dl_tag_thresholds)
pd.DataFrame({"DL v2, global threshold": dl_results, "DL v2, per-tag thresholds": dl_results_per_tag}).T

## 7. Ensemble: ML + DL

The two models make different errors: TF-IDF catches tag names written in the title, the language model catches
meaning. Averaging their scores is cheap: `score = α · ML + (1 − α) · DL`, with α and the threshold tuned on validation.

In [ ]:
def blend(alpha, ml_scores, dl_scores_):
    return alpha * ml_scores + (1 - alpha) * dl_scores_


alpha_search = {}
for alpha in [i / 10 for i in range(11)]:
    s = blend(alpha, ml_val_scores, dl_val_scores)
    alpha_search[alpha] = evaluate(Y_val, s, tune_threshold(Y_val, s))["micro_f1"]
ens_alpha = max(alpha_search, key=alpha_search.get)
print("val micro-F1 by alpha (0 = DL only, 1 = ML only):", {a: round(f, 3) for a, f in alpha_search.items()})

ens_val_scores = blend(ens_alpha, ml_val_scores, dl_val_scores)
ens_threshold = tune_threshold(Y_val, ens_val_scores)
ens_test_scores = blend(ens_alpha, ml_test_scores, dl_test_scores)
ens_results = evaluate(Y_test, ens_test_scores, ens_threshold)
print(f"Chosen alpha = {ens_alpha}")
pd.Series(ens_results, name="Ensemble (test)")

## 8. Comparison on the test set

*DL v1* is the previous run (DistilBERT, same split and seed), copied here as a reference so the effect of the v2
changes is visible without re-training it.

In [ ]:
DL_V1_REFERENCE = {"micro_f1": 0.538, "macro_f1": 0.377, "p@1": 0.656, "r@3": 0.654,
                   "tags/post": np.nan, "threshold": 0.35, "train_time_s": 342.5}

comparison = pd.DataFrame(
    {
        "ML model (TF-IDF + LogReg)": {**ml_results, "train_time_s": ml_train_time},
        "DL v1 (DistilBERT, previous run)": DL_V1_REFERENCE,
        "DL v2 (global threshold)": {**dl_results, "train_time_s": dl_train_time},
        "DL v2 (per-tag thresholds)": {**dl_results_per_tag, "train_time_s": dl_train_time},
        f"Ensemble ML + DL v2 (α={ens_alpha})": {**ens_results, "train_time_s": ml_train_time + dl_train_time},
    }
).T
print(f"True tags per post on test: {Y_test.sum(axis=1).mean():.2f}")
comparison.round(3)

In [ ]:
per_tag = pd.DataFrame(
    {
        "posts_in_test": Y_test.sum(axis=0).astype(int),
        "ML_f1": f1_score(Y_test, predicted_tags_matrix(ml_test_scores, ml_threshold), average=None, zero_division=0),
        "DLv2_f1": f1_score(Y_test, predicted_tags_matrix(dl_test_scores, dl_threshold), average=None, zero_division=0),
        "Ens_f1": f1_score(Y_test, predicted_tags_matrix(ens_test_scores, ens_threshold), average=None, zero_division=0),
    },
    index=kept_tags,
).sort_values("posts_in_test", ascending=False)
per_tag["DLv2_minus_ML"] = per_tag["DLv2_f1"] - per_tag["ML_f1"]
per_tag.round(2).head(25)

## 9. Try it: predict tags for any title

In [ ]:
DEMO_TITLES = [
    "How do I parse JSON in Python?",
    "jQuery click event not firing on dynamically added button",
    "NullReferenceException when binding a GridView in ASP.NET",
    "How to join two tables and group by date in MySQL",
    "UITableView cell height not updating on iPhone",
    "Segmentation fault when using pointers to struct in C",
    "std::vector iterator invalidated after push_back in C++",
]


def predict_all(titles):
    ml = ml_model.predict_proba(titles)
    loader = make_loader(titles, np.zeros((len(titles), len(kept_tags))), shuffle=False)
    dl = dl_scores(dl_model, loader)
    ens = blend(ens_alpha, ml, dl)
    return pd.DataFrame({
        "title": titles,
        "ML model": [top_tags(r, ml_threshold) for r in ml],
        "DL v2": [top_tags(r, dl_threshold) for r in dl],
        "Ensemble": [top_tags(r, ens_threshold) for r in ens],
    })


predict_all(DEMO_TITLES)

## 10. Conclusion

In [ ]:
best = comparison["micro_f1"].astype(float).idxmax()
v1 = DL_V1_REFERENCE
print(f"Best micro-F1 on test: {best} ({comparison.loc[best, 'micro_f1']:.3f})")
print(f"DL v2 vs v1: micro-F1 {dl_results['micro_f1'] - v1['micro_f1']:+.3f}, "
      f"macro-F1 {dl_results['macro_f1'] - v1['macro_f1']:+.3f}, "
      f"tags/post {dl_results['tags/post']:.2f} (truth {Y_test.sum(axis=1).mean():.2f})")
print(f"DL v2 vs ML: micro-F1 {dl_results['micro_f1'] - ml_results['micro_f1']:+.3f}, "
      f"macro-F1 {dl_results['macro_f1'] - ml_results['macro_f1']:+.3f}")
print(f"DL v2 training: {dl_train_time:.0f}s ({dl_train_time / ml_train_time:.1f}x ML), best epoch {best_epoch}")

### v2 conclusion *(to write after the Colab run)*

Questions to answer from the tables above:
* Did the v2 changes fix the v1 symptoms: more than one tag per post, better macro-F1, no "stuck" first epochs?
* Does DL v2 now clearly beat the ML model, i.e. by more than a few hundredths (the noise level on 410 test posts)?
* Do per-tag thresholds help on test, or do they overfit the small validation set?
* Is the ensemble the best option, and is it worth training and serving two models?

### v1 conclusion (previous run, DistilBERT)

#### Results (test set, 410 posts)

| | micro-F1 | macro-F1 | P@1 | R@3 | Training |
|---|---|---|---|---|---|
| ML model (TF-IDF + LogReg) | 0.532 | **0.427** | 0.627 | **0.665** | 37 s, CPU |
| DL model (DistilBERT) | **0.538** | 0.377 | **0.656** | 0.654 | 342 s, T4 GPU |

**1. Overall it is a tie.** DL leads by 0.006 micro-F1 and 0.03 P@1. With only 410 test posts, gaps this small are within
sampling noise (a few hundredths), so neither model is clearly better on frequent tags.

**2. The ML model is better on rare tags (macro-F1 0.427 vs 0.377).** TF-IDF directly picks up the tag name when it appears in
the title (`css`, `string`, `c++`, `xml`), even with few examples. DistilBERT needs more examples per tag. Its uncased
tokenizer also splits `C++` into `c + +` and lowercases `C#`, losing exactly the symbols that identify a technology
(`c++`: 0.48 → 0.25).

**3. The DL model rarely predicts more than one tag.** Three causes add up:
* *The data rewards it*: 55 % of posts have a single kept tag (1.58 on average), so the threshold tuned for micro-F1
  favours predicting one confident tag.
* *Under-training*: validation micro-F1 was still rising at epoch 20 (best epoch = last epoch) while the learning rate had
  decayed to zero. The first 4 epochs were spent learning the trivial "no tag" answer (val micro-F1 ≈ 0.1), typical of
  sigmoid + BCE over 86 mostly-negative labels.
* *Compressed scores*: as a result, secondary tags stay around 0.3–0.45 (e.g. `python` is missing for
  *"How do I parse JSON in Python?"*), just under the 0.35 threshold, whereas the ML model gives them 0.55–0.8.

**4. Cost.** The DL model needs 9× more training time and a GPU, weighs ~268 MB vs a few MB, and is a black box. The ML
model can be explained word by word.

#### Verdict

For this dataset size (≈3.3k training titles), **the ML model is the better PoC choice**. It matches DistilBERT on
frequent tags, beats it on rare tags and on multi-tag predictions, and is far cheaper to train, serve and explain.
The DL model is not a dead end. It is under-trained and badly suited to symbol-heavy tags, which can be fixed.

#### What would improve the DL model (next iteration)
1. Train longer (e.g. 40 epochs) and/or use a higher learning rate for the classification head.
2. Weight positive labels in the loss (`pos_weight`) and tune **one threshold per tag** instead of a global one.
3. Use a **cased or code-aware** pre-trained model (e.g. `bert-base-cased`, `microsoft/codebert-base`) to keep `C#`, `C++`, `.NET`.
4. **Ensemble**: average ML and DL tag scores. The two models make different errors (DL is better on `c#`, `html`, `mysql`,
   ML on `css`, `c++`, `string`), so this is a cheap likely gain.
5. More data: a few thousand posts limit both models to 86 tags. The DL model benefits most from more examples.